# 04 — Analysis

## UAE Used Car Market Analysis

Three things here: a couple of regression models that get progressively
better at predicting price, a test of whether the platform a car's listed
on matters independently of the car itself, and a value-score system that
flags listings priced well below what the model expects for that
make/model/mileage/age combination.

Working from `data/processed/uae_cars_final_clean.csv`, same price-safe
filtering as the EDA notebook.

In [67]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/uae_cars_final_clean.csv")

df_price = df[df["price_aed"].notna() & ~df["price_suspect"]].copy()
df_price = df_price[df_price["mileage_km"].notna() & ~df_price["mileage_suspect"]].copy()
df_price = df_price.dropna(subset=["car_age", "brand_segment"])

print(f"Rows usable for modeling: {len(df_price)}")

Rows usable for modeling: 21922


## Model 1 — mileage, age, and brand segment

The simplest reasonable model: predict price from mileage, car age, and
which of the four brand segments the car falls into. Segment gets
one-hot encoded with Luxury as the baseline, so the other three
segments' coefficients read as "relative to Luxury." An 80/20 train-test
split, so what's reported below is how the model does on cars it hasn't
seen.

In [68]:
segment_dummies = pd.get_dummies(df_price["brand_segment"], prefix="segment")
segment_dummies = segment_dummies.drop(columns=["segment_Luxury"])  # Luxury as baseline

X1 = pd.concat([df_price[["mileage_km", "car_age"]], segment_dummies], axis=1)
y = df_price["price_aed"]

X1_train, X1_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, random_state=42)

model1 = LinearRegression()
model1.fit(X1_train, y_train)
pred1 = model1.predict(X1_test)

r2_1 = r2_score(y_test, pred1)
mae_1 = mean_absolute_error(y_test, pred1)

print(f"R2: {r2_1:.3f}")
print(f"MAE: AED {mae_1:,.0f}")
print()
coef_table = pd.Series(model1.coef_, index=X1.columns).sort_values(ascending=False)
print(coef_table)
print(f"\nIntercept: {model1.intercept_:,.0f}")

R2: 0.091
MAE: AED 63,719

segment_Mid-range      25671.723548
segment_Premium        13617.610495
mileage_km                -0.353295
car_age                -4231.893077
segment_Mass-market   -42196.302368
dtype: float64

Intercept: 191,450


R² of 0.091 — mileage, age, and a four-bucket segment barely
explain any of the price variation on their own. The coefficients show
why: Mid-range (+25,672 AED) and Premium (+13,618 AED) both come out
priced *above* the Luxury baseline once mileage and age are held
constant, which shouldn't happen if "Luxury" cleanly meant "the most
expensive tier." The likely explanation is how broad that bucket
actually is — it's carrying genuine hypercars (Ferrari, Lamborghini)
right alongside low-volume classic and boutique brands (Studebaker,
Borgward, DeLorean) that list for far less than a modern BMW or
Mercedes-Benz. Averaged together, that drags the Luxury baseline down
enough to sit below Mid-range and Premium. Segment alone just isn't a
fine-grained enough category to predict price well across a market this
varied — which is exactly the gap Model 2 is built to close.

## Model 2 — swapping segment for the specific model

Same idea, but instead of "Mass-market vs Premium vs Luxury," the model
gets the actual make+model as a feature. A Land Cruiser and a Yaris being
both "Toyota" and both "Mass-market" tells the segment-level model
nothing useful about the real price gap between them — the specific
model name captures that directly.

Only keeping models with at least 15 listings, so the encoding doesn't
end up with hundreds of one-off dummy columns that are really just
memorizing individual rows.

In [69]:
MIN_LISTINGS = 15

df_price["brand_model"] = df_price["brand"] + " " + df_price["model"].astype(str)
model_counts = df_price["brand_model"].value_counts()
eligible_models = model_counts[model_counts >= MIN_LISTINGS].index

df_model2 = df_price[df_price["brand_model"].isin(eligible_models)].copy()
print(f"{len(eligible_models)} models with {MIN_LISTINGS}+ listings, covering {len(df_model2)} of {len(df_price)} total rows "
      f"({100*len(df_model2)/len(df_price):.1f}%)")

270 models with 15+ listings, covering 18707 of 21922 total rows (85.3%)


In [70]:
model_dummies = pd.get_dummies(df_model2["brand_model"], prefix="model", drop_first=True)

X2 = pd.concat([df_model2[["mileage_km", "car_age"]].reset_index(drop=True), model_dummies.reset_index(drop=True)], axis=1)
y2 = df_model2["price_aed"].reset_index(drop=True)

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X2_train, y2_train)
pred2 = model2.predict(X2_test)

r2_2 = r2_score(y2_test, pred2)
mae_2 = mean_absolute_error(y2_test, pred2)

print(f"R2: {r2_2:.3f}  (Model 1 was {r2_1:.3f})")
print(f"MAE: AED {mae_2:,.0f}  (Model 1 was AED {mae_1:,.0f})")

R2: 0.213  (Model 1 was 0.091)
MAE: AED 44,342  (Model 1 was AED 63,719)


R² roughly doubles, from 0.091 to 0.213, and MAE drops by close
to AED 19,500 (63,719 down to 44,342). A real improvement, and in the
expected direction — the model performs meaningfully better once it
knows it's looking at a Land Cruiser instead of just "a Toyota."

Still, 0.213 on its own is a modest number, and it's worth being honest
about why. "Model" is still a fairly blunt category — a base-trim Range
Rover and a fully-loaded SVR both fall into the same "Land Rover Range
Rover" bucket despite being priced worlds apart, and the dataset's year
range (a handful of listings go back to the 1930s) puts real strain on a
single linear age coefficient. Trim level would probably be the biggest
lever left to pull if there's appetite to push R² further — a limitation
that shows up again later in this notebook.

## Does the platform itself matter?

The EDA notebook turned up something odd: CarSwitch had the highest
median price of the three sources, even though AutoTraders.ae's listings
leaned more toward Premium and Luxury by composition. A chart alone can't
tell you whether that's a real platform effect or just different cars
being sold on different sites — this can.

Adding `source` on top of Model 2's setup, so the model predicts price
from the same make/model/mileage/age as before, plus which site the
listing came from.

In [71]:
source_dummies = pd.get_dummies(df_model2["source"], prefix="source", drop_first=True)

X3 = pd.concat([X2.reset_index(drop=True), source_dummies.reset_index(drop=True)], axis=1)
y3 = y2

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42)

model3 = LinearRegression()
model3.fit(X3_train, y3_train)
pred3 = model3.predict(X3_test)

r2_3 = r2_score(y3_test, pred3)
mae_3 = mean_absolute_error(y3_test, pred3)

print(f"R2: {r2_3:.3f}  (Model 2 without source was {r2_2:.3f})")
print(f"MAE: AED {mae_3:,.0f}  (Model 2 without source was AED {mae_2:,.0f})")
print()
source_coefs = pd.Series(model3.coef_, index=X3.columns).loc[lambda s: s.index.str.startswith("source_")]
print("Source coefficients (relative to the dropped baseline source):")
print(source_coefs)

R2: 0.222  (Model 2 without source was 0.213)
MAE: AED 44,718  (Model 2 without source was AED 44,342)

Source coefficients (relative to the dropped baseline source):
source_carswitch   -29861.250290
source_opensooq    -17157.848123
dtype: float64


R² only moves from 0.213 to 0.222 by adding source — a small
bump, not a big one. But the coefficients are the real result here, and
they flip what the EDA chart suggested. `source_carswitch` comes out at
-29,861 AED and `source_opensooq` at -17,158 AED, both relative to
AutoTraders.ae as the baseline. Once you control for the exact same
model, mileage, and age, CarSwitch listings actually price *lower* than
AutoTraders.ae — the opposite of what the raw median comparison implied.

So the "certified marketplace premium" idea doesn't hold up. What's
really going on looks more like a selection effect — CarSwitch's raw
median came out highest because of *which* cars end up listed there, not
because it charges more for the same car. A more accurate story than the
chart alone suggested, and a good example of why a chart-level pattern is
worth testing before writing it up as a conclusion.

## Value score

Using Model 2's features — model, mileage, age, deliberately without
source, so the "fair price" benchmark doesn't end up favoring or
penalizing a listing just for which platform it's on.

Fitting on **log(price)** rather than raw AED. A straight line on raw
price can predict a negative number for an old, high-mileage car, which
needed an arbitrary floor to patch over. Exponentiating a log-price
model's output can never come out negative in the first place, and it's
a better match for the right-skewed shape Chart 1 showed.

In [72]:
log_full_model = LinearRegression()
log_full_model.fit(X2, np.log(y2))

log_predicted_price = log_full_model.predict(X2)
predicted_price = np.exp(log_predicted_price)

df_model2 = df_model2.reset_index(drop=True)
df_model2["predicted_price"] = predicted_price
df_model2["value_gap_aed"] = df_model2["predicted_price"] - df_model2["price_aed"]
df_model2["value_gap_pct"] = (df_model2["value_gap_aed"] / df_model2["predicted_price"]) * 100

print(f"Predicted price range: AED {predicted_price.min():,.0f} - AED {predicted_price.max():,.0f}")
print("No floor-clipping needed - log-space predictions can't go negative")

Predicted price range: AED 345 - AED 410,477
No floor-clipping needed - log-space predictions can't go negative


In [73]:
pct_underpriced_10 = ((df_model2["value_gap_pct"] > 10) & (df_model2["value_gap_aed"] > 0)).mean() * 100
pct_overpriced_10 = ((df_model2["value_gap_pct"] < -10)).mean() * 100

print(f"Underpriced by 10%+: {pct_underpriced_10:.1f}%")
print(f"Overpriced by 10%+: {pct_overpriced_10:.1f}%")
print(f"\nValue gap % distribution:")
print(df_model2["value_gap_pct"].describe())

Underpriced by 10%+: 44.2%
Overpriced by 10%+: 38.5%

Value gap % distribution:
count    18707.000000
mean       -39.570433
std        480.484417
min     -27577.430694
25%        -33.416688
50%          3.651240
75%         31.351814
max         95.716328
Name: value_gap_pct, dtype: float64


The log transform helped, but only partly. Underpriced-by-10%+
sits at 44.2% and overpriced-by-10%+ at 38.5% — a much more balanced
split than the raw-price version's lopsided 62/28, and the median
(+3.65%) with a roughly symmetric interquartile range (-33% to +31%) both
look genuinely reasonable, close to centered on zero.

What didn't improve: the mean (-39.6%) and minimum (-27,577%) are about
as extreme as before. A median that sane sitting next to a mean that
wild means a small number of outlier rows are still dragging the average
around, even though most of the distribution is behaving fine.

In [74]:
# A gap this extreme only happens when predicted_price is tiny relative
# to actual price - worth seeing which specific rows those are.
extreme_outliers = df_model2.sort_values("value_gap_pct").head(10)
extreme_outliers[["source", "brand_model", "year", "mileage_km", "car_age", "price_aed", "predicted_price", "value_gap_pct"]]

,source,brand_model,year,mileage_km,car_age,price_aed,predicted_price,value_gap_pct
2285,autotraders,Chevrolet Corvette,1974,95000.0,52,300000.0,1083.915640,-27577.430694
2460,autotraders,Nissan SkyLine,2002,73009.0,24,2949995.0,13201.594026,-22245.748507
2461,autotraders,Nissan SkyLine,2001,40749.0,25,2499995.0,12811.525092,-19413.640898
1747,autotraders,Mercedes-Benz S-Class,1971,420.0,55,400000.0,2152.897275,-18479.613839
1734,autotraders,Nissan SkyLine,1999,48.0,27,1749995.0,11561.369487,-15036.571857
7584,autotraders,Porsche 911 Carrera,1994,66000.0,32,1000000.0,6643.499181,-14952.308622
1774,autotraders,Nissan SkyLine,1999,7783.0,27,1679998.0,11402.194256,-14633.988584
1825,autotraders,Nissan SkyLine,2000,87181.0,26,1399995.0,10797.475770,-12865.947133
2420,autotraders,Nissan SkyLine,2001,4335.0,25,1729995.0,13675.559971,-12550.268096
1772,autotraders,Nissan SkyLine,2002,52.0,24,1789995.0,15045.802588,-11796.972524


Every one of these is a classic or collector car — a 1974
Corvette, several late-90s/early-2000s Nissan Skylines (the AED 2.95M
one is almost certainly an R34 GT-R, which genuinely sells for that kind
of money as a collector's item), a 1971 Mercedes S-Class, a 1994 Porsche
911. All 24 to 55 years old.

That's the actual mechanism: a linear age term assumes price keeps
declining forever, but real collector cars don't work that way — value
bottoms out and then climbs back up once a car's old enough to become
desirable again. The model has almost no training examples of a given
model at that age, so it extrapolates toward a price near zero instead
of recognizing the collector-value curve. This isn't a scale problem
log(price) could fix — it's a fundamentally different pricing regime
that a straight line can't represent no matter how the target's
transformed.

The practical fix is scoping the value-score system to the market it's
actually built for: normal depreciation-driven pricing, not the classic
car market, which really deserves its own separate model entirely.

In [75]:
CLASSIC_CAR_AGE_CUTOFF = 25  # beyond this, a linear age term can't represent
                              # the collector-value curve seen above

# The age cutoff alone won't catch everything - Nissan Skyline specifically
# kept showing up in the outlier diagnostic even among cars just under 25
# years old (it's a cult JDM nameplate, collector status kicks in earlier
# than for most models here). Naming it directly rather than just lowering
# the age threshold further, which would start excluding genuinely normal
# older economy cars along with it.
COLLECTOR_NAMEPLATES = ["nissan skyline"]  # matched case-insensitively below -
                                             # the real data has it as "Nissan SkyLine"
                                             # (capital L), a plain == match silently
                                             # caught zero rows on the first attempt

is_collector = df_model2["brand_model"].str.lower().isin(COLLECTOR_NAMEPLATES)
df_scoreable = df_model2[
    (df_model2["car_age"] <= CLASSIC_CAR_AGE_CUTOFF) &
    (~is_collector)
].copy()
print(f"Collector-nameplate rows actually matched: {is_collector.sum()}")
excluded = len(df_model2) - len(df_scoreable)
print(f"Excluding {excluded} listings ({100*excluded/len(df_model2):.1f}% of the modeled set) - "
      f"over {CLASSIC_CAR_AGE_CUTOFF} years old, or a known collector nameplate - {len(df_scoreable)} remain")

pct_underpriced_10 = ((df_scoreable["value_gap_pct"] > 10) & (df_scoreable["value_gap_aed"] > 0)).mean() * 100
pct_overpriced_10 = ((df_scoreable["value_gap_pct"] < -10)).mean() * 100
print(f"\nUnderpriced by 10%+: {pct_underpriced_10:.1f}%")
print(f"Overpriced by 10%+: {pct_overpriced_10:.1f}%")
print(f"\nValue gap % distribution, classics and collector nameplates excluded:")
print(df_scoreable["value_gap_pct"].describe())

Collector-nameplate rows actually matched: 17
Excluding 114 listings (0.6% of the modeled set) - over 25 years old, or a known collector nameplate - 18593 remain

Underpriced by 10%+: 44.4%
Overpriced by 10%+: 38.2%

Value gap % distribution, classics and collector nameplates excluded:
count    18593.000000
mean       -23.236481
std        136.523875
min      -9013.987592
25%        -32.649912
50%          3.990091
75%         31.496052
max         95.716328
Name: value_gap_pct, dtype: float64


In [76]:
# Same diagnostic as before, rerun on the cleaned-up cohort - is there
# another nameable pattern left, or are we into genuine one-off noise now?
remaining_outliers = df_scoreable.sort_values("value_gap_pct").head(10)
remaining_outliers[["source", "brand_model", "year", "mileage_km", "car_age", "price_aed", "predicted_price", "value_gap_pct"]]

,source,brand_model,year,mileage_km,car_age,price_aed,predicted_price,value_gap_pct
1709,autotraders,Mercedes-Benz G-Class,2014,1000.0,12,3900000.0,42791.368330,-9013.987592
2308,autotraders,Rolls Royce Cullinan,2020,13000.0,6,1550000.0,71841.351267,-2057.531801
2309,autotraders,Rolls Royce Cullinan,2025,6000.0,1,2250000.0,112850.946943,-1893.780346
9442,autotraders,BMW X5,2014,245000.0,12,550000.0,27659.787673,-1888.446211
2286,autotraders,Rolls Royce Cullinan,2022,18000.0,4,1680000.0,84870.221945,-1879.492879
16994,opensooq,Toyota Tundra,2024,48000.0,2,1790000.0,95066.534544,-1782.891817
2262,autotraders,Rolls Royce Cullinan,2022,15000.0,4,1600000.0,85327.791938,-1775.121767
1705,autotraders,Rolls Royce Cullinan,2026,0.0,0,2290000.0,124540.883943,-1738.753610
2362,autotraders,Porsche 911 Carrera,2023,900.0,3,1700000.0,95285.229588,-1684.117021
1733,autotraders,Porsche 911 Carrera,2024,0.0,2,1849995.0,104198.755712,-1675.448265


Two different things are mixed into this remaining tail, not one clean
pattern. Rolls-Royce Cullinan and Porsche 911 Carrera show up more than
once each, genuinely priced correctly for what they are (a near-new
Cullinan legitimately sells for 1.5-2.3M AED here) — the model just
doesn't have enough comparable listings at that price tier to calibrate
a sensible coefficient, so it predicts something closer to a normal
SUV/sports car instead. That's a real pattern: thin sample sizes at the
extreme high end break a linear model's ability to price genuine exotics.

The Mercedes G-Class at 3.9M AED, the BMW X5 at 550K, and the Toyota
Tundra at 1.79M are each a single occurrence, and none of those prices
are plausible for those specific models regardless of trim or
condition — these read as data entry errors (an extra digit typed in
somewhere) rather than anything the model should be expected to handle.

That's the point where chasing individual nameplates further stops
making sense — two structurally different problems, neither of which a
single additional exclusion rule would cleanly solve without the list
growing indefinitely. Documenting this rather than patching it further:
**`value_gap_pct` has a genuine long tail worth treating with caution,
especially near the extreme high end of the price distribution where
sample sizes thin out. `value_gap_aed` — already what `good_deals` sorts
by — stays the more trustworthy number**, since an AED gap doesn't blow
up the way a percentage does when the denominator (predicted price) is
small or poorly estimated. Any listing above roughly AED 1M is worth a
manual look at the actual URL before trusting either number outright.

Check the new mean and minimum against the run that only excluded by
age — 107 excluded rows (0.6%) only moved the mean from -39.6% to -27.7%
and barely touched the minimum, because a few Nissan Skylines sitting
right at 24-25 years old survived the cutoff. If naming that nameplate
directly closes the rest of the gap between the mean and the sane
median this run, that confirms it really was Skyline specifically driving
the remaining distortion, not a broader problem with the cutoff choice.

In [77]:
good_deals = df_scoreable[(df_scoreable["value_gap_aed"] > 0) & (df_scoreable["value_gap_pct"] < 80)].copy()
good_deals = good_deals.sort_values("value_gap_aed", ascending=False)

print(f"{len(good_deals)} listings flagged as underpriced out of {len(df_scoreable)} "
      f"({100*len(good_deals)/len(df_scoreable):.1f}%)")
good_deals[["source", "brand_model", "year", "mileage_km", "price_aed", "predicted_price", "value_gap_aed", "value_gap_pct", "url"]].head(20)

9957 listings flagged as underpriced out of 18593 (53.6%)


,source,brand_model,year,mileage_km,price_aed,predicted_price,value_gap_aed,value_gap_pct,url
13378,opensooq,Toyota Land Cruiser,2025,21000.0,189000.0,362081.298474,173081.298474,47.801778,https://ae.opensooq.com/search/283137158
5301,autotraders,Toyota Land Cruiser,2025,100.0,220000.0,375901.831103,155901.831103,41.474081,https://uae.autotraders.ae/used-cars/toyota/la...
13917,opensooq,Toyota Land Cruiser,2025,50.0,225000.0,375935.519221,150935.519221,40.149311,https://ae.opensooq.com/search/282982386
18323,opensooq,Nissan Patrol,2022,500.0,40000.0,180617.111144,140617.111144,77.853704,https://ae.opensooq.com/search/281926080
4694,autotraders,Toyota Land Cruiser,2024,1.0,205000.0,344361.907299,139361.907299,40.469606,https://uae.autotraders.ae/used-cars/toyota/la...
18035,opensooq,Toyota Land Cruiser,2023,30000.0,163000.0,298901.291175,135901.291175,45.466947,https://ae.opensooq.com/search/282100784
3352,autotraders,Toyota Land Cruiser,2025,4000.0,238000.0,373283.437642,135283.437642,36.241479,https://uae.autotraders.ae/used-cars/toyota/la...
3501,autotraders,Toyota Land Cruiser,2024,25000.0,195000.0,329273.012846,134273.012846,40.778627,https://uae.autotraders.ae/used-cars/toyota/la...
4387,autotraders,Toyota Land Cruiser,2026,800.0,278000.0,409888.718917,131888.718917,32.176714,https://uae.autotraders.ae/used-cars/toyota/la...
88,carswitch,Toyota Land Cruiser,2025,950.0,245000.0,375329.594771,130329.594771,34.724039,https://carswitch.com/dubai/used-car/toyota/la...


Worth a skim before trusting this outright. The wide-trim-range
models — Range Rover, Land Cruiser, the Mercedes E/C/S-Class family —
are the ones that showed up dominating this list in earlier runs, and
that's a separate limitation from the classic-car one just fixed: trim
isn't a feature in this model at all, so a base-trim car sitting inside
a wide-trim-range model bucket will still look artificially underpriced
against a prediction blended across configurations it never actually
competed with. If any of those models are still dominating the top of
this list, that's the remaining known limitation, not a new problem.

In [78]:
top_value_models = good_deals["brand_model"].value_counts().head(10)
print("Which models show up most often among the flagged underpriced listings:")
print(top_value_models)

Which models show up most often among the flagged underpriced listings:
brand_model
Nissan Patrol            387
Toyota Land Cruiser      255
Nissan Altima            249
Mercedes-Benz E-Class    180
Mercedes-Benz C-Class    172
Nissan Rogue             165
Dodge Charger            162
Nissan KICKS             158
Nissan Sentra            154
Nissan Sunny             151
Name: count, dtype: int64


If Nissan Patrol and Toyota Land Cruiser keep showing up here,
that lines up with general expectations for this market — genuinely
high-demand vehicles that would plausibly show as a good deal when they
do turn up cheap. Treat the wide-trim-range models on this list with the
caveat above rather than at face value.

## What this is actually useful for

**Dealers** — the models that consistently show up on the flagged
underpriced list (Nissan Patrol and Toyota Land Cruiser being the
clearest examples) are a reasonable starting point for sourcing
inventory. The wide-trim-range models still need a grain of salt, for
the reason noted above — trim isn't in the feature set, so those
predictions are blended across configurations that don't actually
compete on price.

**Lenders** — the jump from Model 1 to Model 2 is the real message for
anyone doing collateral valuation: specific model matters far more than
broad brand tier, and a risk model built only on segment and mileage is
leaving real predictive power on the table. Trim level would likely help
further still, and anything scoring a vehicle over ~25 years old needs a
fundamentally different approach — the collector-car pricing regime
doesn't follow the same depreciation logic at all.

**Insurers** — Chart 9 back in the EDA notebook (Luxury-segment cars
driven noticeably fewer km/year than everything else) is worth folding
directly into usage-based pricing if it isn't already.

**Anyone pricing off a single platform** — this is the one with a real,
tested answer rather than a guess: CarSwitch listings priced *lower*
than AutoTraders.ae for the same model/mileage/age, not higher, despite
having the higher raw median. A pricing tool built only on CarSwitch
data would be systematically underestimating what an equivalent car
sells for elsewhere — the platform itself matters, just not in the
direction the raw numbers first suggested.